# Day 4, Notebook 1: what is typical, and what one order does to it

Yesterday you took fifty orders and decided which forty-four were usable. You wrote down every decision and why you made it.

Today somebody at Kalpa asks what those orders say.

That question has three honest answers and they disagree with each other. This notebook is about picking the one that does not mislead the person who asked.

Everything here runs on the profiled file and nothing else. Describing data you have not cleaned is the mistake this week already made on purpose.

The map below is where this notebook sits in the day and what it adds. Every
notebook in the programme opens on the same pair, so you know where you are before you read a line.

The cell that draws it also brings in the programme's helper. `scripts/c2kit.py` is found by
walking up from this notebook's own folder, which is what lets the same file run whether you
pressed Run All here or a script ran it for you. The helper loads the day's data from `../data/`,
draws every diagram you see in these notebooks, and runs the checks that tell you a cell did what
it claimed.

In [1]:
import sys, pathlib

here = pathlib.Path.cwd()
for parent in [here, *here.parents]:
    if (parent / "scripts" / "c2kit.py").exists():
        sys.path.insert(0, str(parent / "scripts"))
        break
import c2kit as kit

kit.side_by_side(
    kit.ladder(["typical and spread", "the segment summary", "hands-on: the segment summary", "hands-on: repair a broken summary"], lit=0, title="the day's notebooks", show=False),
    kit.flow(["three answers to typical", "the whale", "spread", "the shape off sorted values"], title="what this notebook adds", show=False),
)

## Setup

One cell, at the top, so this notebook runs cold in a fresh Codespace.

`statistics` ships with Python. It is here so you can check your own arithmetic against something you did not write. You compute the median by hand first, then let the library confirm it.

In [2]:
import csv
import statistics

DATA_DIR = "../data"
PROFILED_CSV = f"{DATA_DIR}/C2_W01_D04_profiled_STUDENT.csv"

with open(PROFILED_CSV) as f:
    orders = list(csv.DictReader(f))

# Everything read from a CSV is text. Tuesday's rule, still true.
for r in orders:
    r["amount"] = int(r["amount"])

amounts = [r["amount"] for r in orders]

print("orders loaded:", len(orders))
print("first order:  ", orders[0])

orders loaded: 44
first order:   {'order_id': 'KR4200', 'customer_id': 'C1451', 'segment': 'Student', 'amount': 1280, 'status': 'cancelled', 'order_date': '2026-08-03', 'discount': '150'}


In [3]:
kit.check("forty-four orders came off yesterday's output", len(orders) == 44,
          f"{len(orders)} rows")
kit.check("every amount is a number now, because yesterday rejected the six that were not",
          all(isinstance(a, int) for a in amounts))
kit.check("the smallest order is Rs 800 and the largest is Rs 480,000",
          min(amounts) == 800 and max(amounts) == 480000,
          f"Rs {min(amounts):,} to Rs {max(amounts):,}")

Forty-four orders. That is what yesterday left: fifty went in, six failed conversion and were logged with a reason, forty-four came out.

If your count says anything else, stop and check which file you opened. Every number below rests on this one.

Two things you inherited from yesterday and should keep in view:

- `KR4201` appears **twice**. Yesterday's decision was to keep both rows and flag the pair, because they differ on `order_date` by six weeks and the order book owner decides whether that is a re-export or a repeat order. You did not resolve it, so today you are describing forty-four rows that contain one unresolved pair.
- The **Student** segment held twelve orders in the raw file and holds ten here. Two Student orders failed conversion and were rejected. Your cleaning decisions changed your denominators, which is worth remembering when you report a rate on the smallest segment.

## Where this is going, before we build any of it

One statement about the file you just loaded. It is true.

In [4]:
mean_amount = sum(amounts) / len(amounts)
at_or_above = [a for a in amounts if a >= mean_amount]

print("The average Kalpa order is Rs {:,.2f}.".format(mean_amount))
print("Orders at or above that average:", len(at_or_above), "out of", len(amounts))

The average Kalpa order is Rs 12,753.30.
Orders at or above that average: 1 out of 44


An average that describes one order out of forty-four.

Nothing crashed. The arithmetic is correct. Send that first line to somebody who trusts you and they will plan a quarter around a number that fits one customer.

The gap between a correct number and an honest description is the whole of today.

In [5]:
kit.flow(["three answers to typical", "the whale", "spread", "the shape off sorted values"], lit=0)

## Section 1: three answers to "what is typical"

Before any code, do this on paper. Seven values, which are the amounts of the first seven orders in your profiled file.

```
1280   1865   2270   2835   1310   1145   1030
```

Three ways to answer "what does a typical one look like":

```
  MEAN     add them all, share the total out equally
             |
  MEDIAN   stand them in a line, walk to the middle, read that one
             |
  MODE     which exact value shows up most often
```

Work out the mean and the median with a pen. Then run the next cell.

In [6]:
seven = amounts[:7]
print("the seven:", seven)
print("sum:      ", sum(seven))
print("mean:     ", round(sum(seven) / len(seven), 2))
print("sorted:   ", sorted(seven))
print("median:   ", statistics.median(seven))

the seven: [1280, 1865, 2270, 2835, 1310, 1145, 1030]
sum:       11735
mean:      1676.43
sorted:    [1030, 1145, 1280, 1310, 1865, 2270, 2835]
median:    1310


In [7]:
kit.check("seven values, taken in file order rather than sorted", len(seven) == 7)
kit.check("their mean and median sit within Rs 400 of each other",
          abs(sum(seven) / 7 - sorted(seven)[3]) < 400,
          f"mean Rs {sum(seven) / 7:,.2f} against median Rs {sorted(seven)[3]:,}")

### Three answers to one question

In [8]:
kit.flow(["mean: share the total out equally",
          "median: the order standing in the middle",
          "mode: the exact value that repeats most"], lit=1,
         title="what is typical, three ways")

Mean Rs 1,676.43, median Rs 1,310. Close enough that either one describes these seven orders honestly.

The mean used all seven values. The median used their order and the one value standing in the middle. That difference does nothing here, which is exactly why the next cell matters.

### One order changes, and the two answers part company

Keep the same seven orders. Replace the largest of them, Rs 2,835, with the largest order in the real file: Rs 480,000. Change nothing else.

In [9]:
biggest_in_file = max(amounts)
seven_sorted = sorted(seven)
seven_with_the_whale = seven_sorted[:-1] + [biggest_in_file]

print("original: ", seven_sorted)
print("  mean:   ", round(sum(seven_sorted) / 7, 2))
print("  median: ", statistics.median(seven_sorted))
print()
print("changed:  ", seven_with_the_whale)
print("  mean:   ", round(sum(seven_with_the_whale) / 7, 2))
print("  median: ", statistics.median(seven_with_the_whale))

original:  [1030, 1145, 1280, 1310, 1865, 2270, 2835]
  mean:    1676.43
  median:  1310

changed:   [1030, 1145, 1280, 1310, 1865, 2270, 480000]
  mean:    69842.86
  median:  1310


In [10]:
kit.check("one substitution moves the mean by more than forty times",
          sum(seven_with_the_whale) / 7 / (sum(seven) / 7) > 40)
kit.check("and the median moves by less than a thousand rupees",
          abs(statistics.median(seven_with_the_whale) - statistics.median(seven)) < 1000)

### Why one moves and the other does not

In [11]:
kit.matrix(["mean", "median"],
           ["what it uses", "what one huge value does"],
           [["every value in the column", "drags it, in proportion to how huge"],
            ["only the position in the middle", "nothing, unless it changes who is in the middle"]],
           title="the same seven orders, one value replaced")

The mean went from Rs 1,676.43 to Rs 69,842.86, which is nearly forty-two times larger. The median did not move by one rupee.

That is not a quirk of these seven numbers. It is what the two statistics are built to do:

- The **mean** gives every order a vote, weighted by size. A large order shouts.
- The **median** gives every order a vote of equal weight. A large order is one more order standing to the right.

Choosing between them is choosing whether the loudest order speaks for the quiet ones.

### Mode, and why it earns little here

In [12]:
counts_by_amount = {}
for a in amounts:
    counts_by_amount[a] = counts_by_amount.get(a, 0) + 1

repeated = sorted(counts_by_amount.items(), key=lambda pair: pair[1], reverse=True)[:3]
print("the three most repeated amounts:", repeated)
print("total orders:", len(amounts))

the three most repeated amounts: [(1865, 2), (2895, 2), (2050, 2)]
total orders: 44


The most repeated amount appears twice out of forty-four.

Mode answers "which exact value repeats", and on a money column almost nothing repeats. Ask it about a category and it becomes useful at once: "the most common segment" or "the most common status" are answers a stakeholder can act on.

The rule worth keeping: mode is for categories, and `amount` is not a category.

In [13]:
kit.flow(["three answers to typical", "the whale", "spread", "the shape off sorted values"], lit=1)

## Section 2: the whale

Run the mean on the real column again, this time with the two lines that expose it.

In [14]:
mean_amount = sum(amounts) / len(amounts)
above = [a for a in amounts if a >= mean_amount]
below = [a for a in amounts if a < mean_amount]

print("mean amount over {} orders: Rs {:,.2f}".format(len(amounts), mean_amount))
print("orders at or above Rs {:,.2f}: {}".format(mean_amount, len(above)))
print("orders below Rs {:,.2f}: {}".format(mean_amount, len(below)))

mean amount over 44 orders: Rs 12,753.30
orders at or above Rs 12,753.30: 1
orders below Rs 12,753.30: 43


### The deliberate failure of this half, and it never raises

```
mean amount over 44 orders: Rs 12,753.30
orders at or above Rs 12,753.30: 1
orders below Rs 12,753.30: 43
```

No traceback. No red text. Nothing in Tuesday's toolkit fires, because nothing went wrong in the sense Python understands.

The failure is that a true sentence produced a false impression. `try` and `except` cannot catch that. The only thing that catches it is looking at the shape before you speak.

Sort the column and the cause walks out on its own.

In [15]:
print("the seven largest orders in the file:")
for a in sorted(amounts)[-7:]:
    print("   Rs {:>9,}".format(a))

the seven largest orders in the file:
   Rs     2,855
   Rs     2,895
   Rs     2,895
   Rs     2,930
   Rs     2,990
   Rs     2,995
   Rs   480,000


The second largest order at Kalpa is Rs 2,995. The largest is Rs 480,000, which is a hundred and sixty times the one below it.

In [16]:
whale_amount = max(amounts)
whale = [r for r in orders if r["amount"] == whale_amount][0]

print("the whale:", whale)
print()
print("it is {:.1f} percent of all revenue in this file".format(100 * whale_amount / sum(amounts)))

the whale: {'order_id': 'KR4232', 'customer_id': 'C1749', 'segment': 'Retail-Core', 'amount': 480000, 'status': 'delivered', 'order_date': '2026-08-19', 'discount': ''}

it is 85.5 percent of all revenue in this file


One order. `KR4232`, a delivered Retail-Core order, and it carries eighty-five and a half percent of every rupee in the file.

Yesterday's pass looked at it and kept it, and the decisions log records why: it converts cleanly and is well formed, so it is real until somebody says otherwise, and it was raised with the order book owner.

**An order can be perfectly correct and still wreck every summary it touches.**

In [17]:
ordinary = [a for a in amounts if a != whale_amount]

print("with the whale:    n={:2d}  mean=Rs {:>10,.2f}  median=Rs {:>8,.2f}".format(
    len(amounts), sum(amounts) / len(amounts), statistics.median(amounts)))
print("without the whale: n={:2d}  mean=Rs {:>10,.2f}  median=Rs {:>8,.2f}".format(
    len(ordinary), sum(ordinary) / len(ordinary), statistics.median(ordinary)))

with the whale:    n=44  mean=Rs  12,753.30  median=Rs 1,910.00
without the whale: n=43  mean=Rs   1,887.09  median=Rs 1,865.00


In [18]:
kit.check("the mean over 44 orders is about Rs 12,753",
          round(sum(amounts) / len(amounts)) == 12753, f"Rs {sum(amounts) / len(amounts):,.2f}")
kit.check("only one order sits at or above it", len(above) == 1, f"{len(above)} of 44")
kit.check("taking that one out drops the mean below Rs 1,900",
          sum(ordinary) / len(ordinary) < 1900,
          f"Rs {sum(ordinary) / len(ordinary):,.2f} across {len(ordinary)} orders")
kit.check("KR4232 is 85 percent or more of the money in the file",
          whale_amount / sum(amounts) > 0.85,
          f"{100 * whale_amount / sum(amounts):.1f} percent")

### The tail, drawn

In [19]:
kit.decision_ladder([f"Rs {a:,}" for a in sorted(amounts)[-5:]], cut_at=4,
                    title="the five largest orders, and where the file stops being ordinary")

Look at the second row. Take one order out of forty-four and the mean falls from Rs 12,753.30 to Rs 1,887.09, landing within Rs 22 of the median.

That is the tell. When removing a single record makes the mean and the median agree, the mean was describing that record rather than the business.

The median moved by Rs 45 across the same change.

Now the question that decides whether you are an analyst or a decorator: **do you delete the whale?**

No. Yesterday you wrote a decisions log precisely so nobody could quietly remove an inconvenient order. The order stays in the file and the statistic changes. Deleting real data to make a number tidier is how a report turns into fiction, and it surfaces six months later in front of people who did not do it.

### Milestone 1: what you can now answer

**Where this shows up in production.** National statistical agencies report *median* household income rather than the mean, because a small number of very high incomes drag the mean away from anything a household would recognise. The same convention runs through salary bands, insurance claim sizes, invoice amounts and basket totals. Any team reporting "average order value" on a marketplace without checking the tail first has shipped this bug.

**Interview question this section just made answerable.**

> A stakeholder asks for the average order value. One enormous order sits in the data. What number do you give them, and what do you say?

A complete answer has three parts. Give the median and name it as the median. State the count it rests on. Say the large order exists, that it is real and retained, and that quoting the mean would describe one order out of forty-four. Volunteering the third part is what separates a good answer from a correct one.

In [20]:
kit.flow(["three answers to typical", "the whale", "spread", "the shape off sorted values"], lit=2)

## Section 3: spread, and a rule that runs without you

Typical is one number. Spread is how far the file wanders from it.

In [21]:
print("min:   Rs {:>9,}".format(min(amounts)))
print("max:   Rs {:>9,}".format(max(amounts)))
print("range: Rs {:>9,}".format(max(amounts) - min(amounts)))

min:   Rs       800
max:   Rs   480,000
range: Rs   479,200


Range is one subtraction and the least stable number you will produce today. It is built from exactly two orders out of forty-four, and one of them is the whale. A statistic computed from two orders tells you about those two orders.

Yesterday you flagged the whale with a fence set at ten times the middle order. That worked and it was deliberately crude. Here is the standard version, which uses the spread of the middle half rather than a multiplier somebody chose.

```
   Q1                    Q3
    |                     |
 ---+---------------------+-------------------------|
    |<------ IQR -------->|                    upper fence
                                            Q3 + 1.5 x IQR
```

In [22]:
q1, q2, q3 = statistics.quantiles(amounts, n=4)
iqr = q3 - q1
upper_fence = q3 + 1.5 * iqr

print("Q1:          Rs {:>9,.2f}".format(q1))
print("Q3:          Rs {:>9,.2f}".format(q3))
print("IQR:         Rs {:>9,.2f}".format(iqr))
print("upper fence: Rs {:>9,.2f}".format(upper_fence))
print()
flagged = [a for a in amounts if a > upper_fence]
print("orders above the fence:", len(flagged), "->", flagged)

Q1:          Rs  1,287.50
Q3:          Rs  2,718.75
IQR:         Rs  1,431.25
upper fence: Rs  4,865.62

orders above the fence: 1 -> [480000]


In [23]:
kit.check("the fence catches exactly one order",
          len([a for a in amounts if a > upper_fence]) == 1)
kit.check("and it is the whale", max(amounts) > upper_fence)
kit.check("the fence is a convenience rather than a test",
          upper_fence < max(amounts), f"fence at Rs {upper_fence:,.0f}")

The fence catches exactly one order, and it is the one yesterday's cruder rule caught. The largest ordinary order at Rs 2,995 sits well inside it.

Two rules, built differently, agreeing on the same single record. That agreement is worth more than either rule on its own.

**A fence is a flag, never a delete key.** It says "look at this order". Yesterday's language holds without change: an outlier is a finding to investigate before it is a row to delete. The fence only finds it faster, on a file too large to eyeball.

In [24]:
kit.flow(["three answers to typical", "the whale", "spread", "the shape off sorted values"], lit=3)

## Section 4: reading the shape off sorted values

You have no charting library until Week 2. You do not need one for this.

Sort the column. Stand on the median. Look both ways.

```
 min       median                                           max
  |           |                                              |
  +-----------+----------------------------------------------+
   distance down            distance up
```

If the two distances are similar, the shape is roughly even. If one is far longer, that side has a tail, and the tail is what pulls the mean.

In [25]:
median_amount = statistics.median(amounts)
down = median_amount - min(amounts)
up = max(amounts) - median_amount

print("median:        Rs {:>10,.2f}".format(median_amount))
print("distance down: Rs {:>10,.2f}".format(down))
print("distance up:   Rs {:>10,.2f}".format(up))
print("the up side is {:,.0f} times the down side".format(up / down))
print()
print("mean / median ratio: {:.2f}".format(mean_amount / median_amount))

median:        Rs   1,910.00
distance down: Rs   1,110.00
distance up:   Rs 478,090.00
the up side is 431 times the down side

mean / median ratio: 6.68


The tell you can use in any interview and on any dataset, with no formula at all:

| What you see | What it means |
|---|---|
| Mean noticeably larger than median | Something large is pulling on the right |
| Mean noticeably smaller than median | Something small is pulling on the left |
| Mean and median close together | The shape is roughly even, either one describes it |

On this file the mean is 6.68 times the median. You knew there was a tail before you looked at a single order.

Run those two numbers first on any new column. It costs one line and it tells you which statistic you are allowed to quote.

### Milestone 2: what you can now answer

**Where this shows up in production.** In 1973 the statistician Frank Anscombe built four datasets sharing nearly identical means, variances and correlation coefficients that look nothing like each other when drawn. The set is still handed to new analysts for one reason: a summary statistic is a compression, and every compression throws something away. Two datasets can hand you the same numbers and describe two different worlds.

Until Week 2 gives you a plotting library, the sorted list and the mean-to-median ratio are your picture. They are cheap and they catch most of what a chart would have shown you.

**Interview question this section just made answerable.**

> How would you check for skew without plotting anything?

Sort the values, take the median, and compare the distance from the median to the maximum against the distance from the median to the minimum. Then compare the mean against the median: a mean well above the median means a right tail. Say the second part even if they only asked for one, because it is one line of code on any column in any language.

### What this notebook established

In [26]:
kit.table(
    ["The idea", "What proved it here"],
    [["Mean and median answer the same question differently", "Rs 12,753 against Rs 1,910 on the same 44 orders"],
     ["One record can own the mean", "KR4232 is 85 percent of the money and one order sits at or above the mean"],
     ["Range is built from the two least typical values", "Rs 800 to Rs 480,000, and the fence catches one row"],
     ["Skew reads off sorted values with no chart", "the distance up from the median dwarfs the distance down"]],
    caption="Day 4, notebook 1",
)
kit.flow(["three answers to typical", "the whale", "spread", "the shape off sorted values"], lit=3, title="the notebook, end to end")
kit.check_summary()

The idea,What proved it here
Mean and median answer the same question differently,"Rs 12,753 against Rs 1,910 on the same 44 orders"
One record can own the mean,KR4232 is 85 percent of the money and one order sits at or above the mean
Range is built from the two least typical values,"Rs 800 to Rs 480,000, and the fence catches one row"
Skew reads off sorted values with no chart,the distance up from the median dwarfs the distance down


## What this notebook settled

| Question | Answer |
|---|---|
| Typical Kalpa order, honest version | Median, Rs 1,910 on 44 orders |
| Typical Kalpa order, misleading version | Mean, Rs 12,753.30, which describes 1 order out of 44 |
| Why they differ | One real order, `KR4232`, at Rs 480,000 |
| Does the whale get deleted | No. It is real, and the decisions log says why it stays. |
| Shape, with no chart | Mean is 6.68 times the median, so there is a right tail |

**Crux.** The mean was right and the description was wrong. On a money column, send the median and say that is what you sent.

Notebook 2 takes this one level down, to the segment, where a second thing starts lying: the denominator.